<a href="https://colab.research.google.com/github/danielalopez-lang/Tesis_Maestria-Resistoma_Chakras-Tena-/blob/main/Analisis_in_silico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Instalar Biopython primero mediante pip
!pip install --quiet biopython

# 2. Actualizar paquetes de APT e instalar ncbi-blast+
!apt-get update --allow-insecure-repositories -qq || true
!apt-get install -y ncbi-blast+ -qq

import os
import glob
import pandas as pd
from Bio import SeqIO

print("Entorno configurado y dependencias instaladas correctamente")

In [ ]:
# 2.DEFINIR CONTENIDO GENES
contenido_genes = """
>Sul2
ATGAATAAATCGCTCATCATTTTCGGCATCGTCAACATAACCTCGGACAGTTTCTCCGATGGAGGCCGGT
ATCTGGCGCCAGACGCAGCCATTGCGCAGGCGCGTAAGCTGATGGCCGAGGGGGCAGATGTGATCGACCT
CGGTCCGGCATCCAGCAATCCCGACGCCGCGCCTGTTTCGTCCGACACAGAAATCGCGCGTATCGCGCCG
GTGCTGGACGCGCTCAAGGCAGATGGCATTCCCGTCTCGCTCGACAGTTATCAACCCGCGACGCAAGCCT
ATGCCTTGTCGCGTGGTGTGGCCTATCTCAATGATATTCGCGGTTTTCCAGACGCTGCGTTCTATCCGCA
ATTGGCGAAATCATCTGCCAAACTCGTCGTTATGCATTCGGTGCAAGACGGGCAGGCAGATCGGCGCGAG
GCACCCGCTGGCGACATCATGGATCACATTGCGGCGTTCTTTGACGCGCGCATCGCGGCGCTGACGGGTG
CCGGTATCAAACGCAACCGCCTTGTCCTTGATCCCGGCATGGGGTTTTTTCTGGGGGCTGCTCCCGAAAC
CTCGCTCTCGGTGCTGGCGCGGTTCGATGAATTGCGGCTGCGCTTCGATTTGCCGGTGCTTCTGTCTGTT
TCGCGCAAATCCTTTCTGCGCGCGCTCACAGGCCGTGGTCCGGGGGATGTCGGGGCCGCGACACTCGCTG
CAGAGCTTGCCGCCGCCGCAGGTGGAGCTGACTTCATCCGCACACACGAGCCGCGCCCCTTGCGCGACGG
GCTGGCGGTATTGGCGGCGCTGAAAGAAACCGCAAGAATTCGTTAA

>Sul1
ATGGTGACGGTGTTCGGCATTCTGAATCTCACCGAGGACTCCTTCTTCGATGAGAGCCGGCGGCTAGACC
CCGCCGGCGCTGTCACCGCGGCGATCGAAATGCTGCGAGTCGGAGCAGACGTCGTGGATGTCGGACCGGC
CGCCAGCCATCCGGACGCGAGGCCTGTATCGCCGGCCGATGAGATCAGACGTATTGCGCCGCTCTTAGAC
GCCCTGTCCGATCAGATGCACCGTGTTTCAATCGACAGCTTCCAACCGGAAACCCAGCGCTATGCGCTCA
AGCGCGGCGTGGGCTACCTGAACGATATCCAAGGATTTCCTGACCCTGCGCTCTATCCCGATATTGCTGA
GGCGGACTGCAGGCTGGTGGTTATGCACTCAGCGCAGCGGGATGGCATCGCCACCCGCACCGGTCACCTT
CGACCCGAAGACGCGCTCGACGAGATTGTGCGGTTCTTCGAGGCGCGGGTTTCCGCCTTGCGACGGAGCG
GGGTCGCTGCCGACCGGCTCATCCTCGATCCGGGGATGGGATTTTTCTTGAGCCCCGCACCGGAAACATC
GCTGCACGTGCTGTCGAACCTTCAAAAGCTGAAGTCGGCGTTGGGGCTTCCGCTATTGGTCTCGGTGTCG
CGGAAATCCTTCTTGGGCGCCACCGTTGGCCTTCCTGTAAAGGATCTGGGTCCAGCGAGCCTTGCGGCGG
AACTTCACGCGATCGGCAATGGCGCTGACTACGTCCGCACCCACGCGCCTGGAGATCTGCGAAGCGCAAT
CACCTTCTCGGAAACCCTCGCGAAATTTCGCAGTCGCGACGCCAGAGACCGAGGGTTAGATCATGCCTAG

>blaTEM-1
ATGAGTATTCAACATTTTCGTGTCGCCCTTATTCCCTTTTTTGCGGCATTTTGCCTTCCTGTTTTTGCTC
ACCCAGAAACGCTGGTGAAAGTAAAAGATGCTGAAGATCAGTTGGGTGCACGAGTGGGTTACATCGAACT
GGATCTCAACAGCGGTAAGATCCTTGAGAGTTTTCGCCCCGAAGAACGTTTTCCAATGATGAGCACTTTT
AAAGTTCTGCTATGTGGTGCGGTATTATCCCGTGTTGACGCCGGGCAAGAGCAACTCGGTCGCCGCATAC
ACTATTCTCAGAATGACTTGGTTGAGTACTCACCAGTCACAGAAAAGCATCTTACGGATGGCATGACAGT
AAGAGAATTATGCAGTGCTGCCATAACCATGAGTGATAACACTGCTGCCAACTTACTTCTGACAACGATC
GGAGGACCGAAGGAGCTAACCGCTTTTTTGCACAACATGGGGGATCATGTAACTCGCCTTGATCGTTGGG
AACCGGAGCTGAATGAAGCCATACCAAACGACGAGCGTGACACCACGATGCCTGCAGCAATGGCAACAAC
GTTGCGCAAACTATTAACTGGCGAACTACTTACTCTAGCTTCCCGGCAACAATTAATAGACTGGATGGAG
GCGGATAAAGTTGCAGGACCACTTCTGCGCTCGGCCCTTCCGGCTGGCTGGTTTATTGCTGATAAATCTG
GAGCCGGTGAGCGTGGGTCTCGCGGTATCATTGCAGCACTGGGGCCAGATGGTAAGCCCTCCCGTATCGT
AGTTATCTACACGACGGGGAGTCAGGCAACTATGGATGAACGAAATAGACAGATCGCTGAGATAGGTGCC
TCACTGATTAAGCATTGGTAA

>tetC
ATGGAAAATAAAAATCATCAACAAGAAAATTTTAAGAGTACCTATCAATCACTGGTTAACTCAGCACGAA
TATTGTTTGTTGAAAAAGGCTATCAAGCTGTTTCAATAGATGAGATCTCGGGAAAAGCGTTGGTGACCAA
AGGTGCCTTTTATCATCACTTTAAAAATAAAAAACAATTACTCAGTGCCTGTTATAAGCAGCAATTAATT
ATGATTGATGCCTACATCACAACAAAAACTGATTTAACAAATGGTTGGTCTGCCTTAGAAAGTATATTTG
AACATTATCTTGATTATATTATTGATAATAATAAAAACCTTATCCCTATCCAAGAAGTGATGCCTATCAT
TGGTTGGAATGAACTTGAAAAAATTAGCCTTGAATACATTACTGGTAAGGTAAACGCCATTGTCAGCAAA
TTGATCCAAGAGAACCAACTTAAAGCTTATGATGATGATGTGCTTAAAAACTTACTCAATGGCTGGTTTA
TGCATATCGCAATACATGCGAAAAACCTAAAAGAGCTTGCCGATAAAAAAGGCCAATTTATTGCTATTTA
CCGCGGCTTTTTATTGAGCTTGAAAGATAAATAA

>floR
ATGACCACCACACGCCCCGCGTGGGCCTATACGCTGCCGGCAGCACTGCTGCTGATGGCTCCTTTCGACA
TCCTCGCTTCACTGGCGATGGATATTTATCTCCCTGTCGTTCCAGCGATGCCCGGCATCCTGAACACGAC
GCCCGCTATGATCCAACTCACGTTGAGCCTCTATATGGTGATGCTCGGCGTGGGCCAGGTGATTTTTGGT
CCGCTCTCAGACAGAATCGGGCGACGGCCAATTCTACTTGCGGGCGCAACGGCTTTCGTCATTGCGTCTC
TGGGAGCAGCTTGGTCTTCAACTGCACCGGCCTTTGTCGCTTTCCGTCTACTTCAAGCAGTGGGCGCGTC
GGCCATGCTGGTGGCGACGTTCGCGACGGTTCGCGATGTTTACGCCAGCCGGCCCGAGGGCGTCGTCATC
TACGGTCTTTTCAGTTCGATGCTGGCCTTCGTGCCTGCGCTTGGCCCAATTGCTGGAGCACTGATCGGCG
AGTTTTTCGGATGGCGGGCGATATTCGTAACGCTGGCCGCGCTTGCGTTACCCGCACTCTTGAATGCCGG
TTTCAGGTGGCACGAAACCCGCCCTCTGGATCAAGTCAAGACGCGCCGATCTGTCTTGCCGATCTTCGCG
AGTCCGGCTTTTTGGGTTTACACTGTCGGCTTTAGCGCCGGTATGGGCACCTTCTTCGTCTTCTTCTCGA
CGGCTCCCCGTGTGCTCATAGGCCAAGCGGAATATTCCGAGATCGGATTCAGCTTTGCCTTCGCCACTGT
CGCGCTTGTAATGATCGTGACAACCCGTTTCGCGAAGTCCTTTGTCGCCAGATGGGGCATCGCAGGATGC
GTGGCGCGTGGGATGGCGTTGCTTGTTTGCGGAGCGGTCCTGTTGGGGATCGGCGAACTTTACGGCTCGC
CGTCATTCCTCACCTTCATCCTACCGATGTGGGTTGTCGCGGTCGGTATTGTCTTCACGGTGTCCGTTAC
CGCGAACGGCGCTTTGGCAGAGTTCGACGACATCGCGGGATCAGCGGTCGCGTTCTACTTCTGCGTTCAA
AGCCTGATAGTCAGCATTGTCGGGACATTGGCGGTGGCACTTTTAAACGGTGACACAGCGTGGCCCGTGA
TCTGTTACGCCACGGCGATGGCGGTACTGGTTTCGTTGGGGCTGGTGCTCCTTCGGCTCCGTGGGGCTGC
CACCGAGAAGTCGCCAGTCGTCTAA

>cmlA
GTGCGCTCAAAAAATTTTAGTTGGCGGTACTCCCTTGCCGCCACGGTGTTGTTGTTATCACCGTTCGATT
TATTGGCATCACTCGGCATGGACATGTACTTGCCGGCAGTGCCGTTTATGCCAAACGCGCTTGGTACGAC
AGCGAGCACAATTCAGCTTACGCTGACAACGTACTTGGTCATGATTGGTGCCGGTCAGCTCTTGTTTGGA
CCGCTATCGGACCGACTGGGGCGCCGCCCCGTTCTACTGGGAGGTGGCCTCGCCTACGTTGTGGCGTCAA
TGGGCCTCGCTCTTACGTCATCGGCTGAAGTCTTTCTGGGGCTTCGGATTCTTCAGGCTTGTGGTGCCTC
GGCGTGCCTTGTTTCCACGTTTGCAACAGTACGTGACATTTACGCAGGTCGCGAGGAAAGTAATGTCATT
TACGGCATACTCGGATCCATGCTGGCCATGGTCCCGGCGGTAGGCCCATTGCTCGGAGCGCTCGTCGACA
TGTGGCTTGGGTGGCGGGCTATCTTTGCGTTTCTAGGTTTGGGCATGATCGCTGCATCTGCAGCAGCGTG
GCGATTCTGGCCTGAAACCCGGGTGCAACGAGTTGCGGGCTTGCAATGGTCGCAGCTGCTACTCCCCGTT
AAGTGCCTGAACTTCTGGTTGTACACGTTGTGTTACGCCGCTGGAATGGGTAGCTTCTTCGTCTTTTTCT
CCATTGCGCCCGGACTAATGATGGGCAGGCAAGGTGTGTCTCAGCTTGGCTTCAGCCTGCTGTTCGCCAC
AGTGGCAATTGCCATGGTGTTTACAGCTCGTTTTATGGGGCGTGTAATACCCAAGTGGGGCAGCCCAAGC
GTCTTGCGAATGGGAATGGGATGCCTGATAGCTGGAGCAGTATTGCTTGCCATCACCGAAATATGGGCTT
CGCAGTCCGTGTTAGGCTTTATTTCTCCAATGTGGCTAGTGGGTATTGGTGTCGCCACAGCGGTATCTGT
GGCGCCCAATGGCGCTCTTCGAGGATTCGACCATGTTGCTGGAACGGTCACGGCAGTCTACTTCTGCTTG
GGCGGTGTACTGCTAGGAAGCATCGGAACGTTGATCATTTCGCTGTTGCCGCGCAACACGGCTTGGCCGG
TTGTCGTGTACTGTTTGACCCTTGCAACAGTCGTGCTCGGTCTGTCTTGTGTTTCCCGAGCGGAGGGCTC
TCGCGGCCAGGGGGAGCATGATGTGGTCGCGCTACAAAGTGCGGAAAGTACGTCAAATCCCAATCGTTGA

>blaOXA-1
ATGAAAAACACAATACATATCAACTTCGCTATTTTTTTAATAATTGCAAATATTATCTACAGCAGCGCCA
GTGCATCAACAGATATCTCTACTGTTGCATCTCCATTATTTGAAGGAACTGAAGGTTGTTTTTTACTTTA
CGATGCATCCACAAACGCTGAAATTGCTCAATTCAATAAAGCAAAGTGTGCAACGCAAATGGCACCAGAT
TCAACTTTCAAGATCGCATTATCACTTATGGCATTTGATGCGGAAATAATAGATCAGAAAACCATATTCA
AATGGGATAAAACCCCCAAAGGAATGGAGATCTGGAACAGCAATCATACACCAAAGACGTGGATGCAATT
TTCTGTTGTTTGGGTTTCGCAAGAAATAACCCAAAAAATTGGATTAAATAAAATCAAGAATTATCTCAAA
GATTTTGATTATGGAAATCAAGACTTCTCTGGAGATAAAGAAAGAAACAACGGATTAACAGAAGCATGGC
TCGAAAGTAGCTTAAAAATTTCACCAGAAGAACAAATTCAATTCCTGCGTAAAATTATTAATCACAATCT
CCCAGTTAAAAACTCAGCCATAGAAAACACCATAGAGAACATGTATCTACAAGATCTGGATAATAGTACA
AAACTGTATGGGAAAACTGGTGCAGGATTCACAGCAAATAGAACCTTACAAAACGGATGGTTTGAAGGGT
TTATTATAAGCAAATCAGGACATAAATATGTTTTTGTGTCCGCACTTACAGGAAACTTGGGGTCGAATTT
AACATCAAGCATAAAAGCCAAGAAAAATGCGATCACCATTCTAAACACACTAAATTTATAA

>gshA
TTGATCCCGGACGTATCTCAGGCGCTGGCCTGGCTGGAAAACCACCCTCAGGCATTGAAGGGTATTCAGC
GTGGTCTTGAGCGTGAAACGCTGCGCGTTAACGCGGACGGTAGCTTAGCGACGACGGGTCACCCGAAGGC
GTTAGGCTCGGCGCTGACACATAAATGGATCACAACCGATTTCGCTGAAGCACTACTGGAGTTCATCACG
CCAGTAGAGGGTGATATTGATCATATGCTGACGATTATGCGCGATATCCATCGCTACACCGCGCGCAACA
TGGGCGACGAACGTATGTGGCCCCTCAGCATGCCGTGCTATATCGAGCAGGGCCAGGAGATTGAGCTGGC
GCAGTACGGCACGTCAAATATCGGCCGGCTGAAAACGCTCTATCGCGAAGGGCTGAAAAACCGCTACGGC
GCGTTGATGCAGACGATCTCGGGCGTGCATTATAACTTCTCGCTGCCAATGGCGTTCTGGCAGGCGAAAT
GCGGTGAAACGGATAAAGATGCGATCTCTGAAGGCTACTTCCGCCTGATCCGCAACTATTACCGTTTCGG
CTGGGTGATCCCGTATCTGTTTGGGGCCTCCCCGGCGATCTGTTCTTCGTTCCTGCAGGGGAAACCGACC
ACGCTGCCGTTCGAGAAGACCGAGTGCGGAATGTACTATCTCCCGTACGCGACCTCTCTGCGCCTGAGCG
ATCTTGGTTATACCAATAAATCGCAAAGCAATCTCGGTATTACGTTTAACGAATTGCACGAATATGTGGC
AGGATTGAAGCGGGCGATCAAAACCCCGTCGGAAGAGTACGAGAAAATCGGCCTCGAAAAAGACGGCAAG
CGCCTGCAAATCAACAGCAATGTGCTGCAGATTGAAAACGAGCTGTATGCGCCTATTCGTCCTAAGCGCG
TAACGCGCAGCGGTGAAACCCCGTCGGATGCGCTGCAGCGCGGCGGGATCGAATACATTGAAGTGCGTTC
GCTGGATATCAACCCGTTCTCACCGATTGGCGTTGATGAGCAGCAGATTCGCTTCCTGGATCTGTTTATG
GTCTGGTGCGTGCTGGCGGATGCGCCGGAAATGAGTTCAGACGAACTGCTCTGTACCCGTACAAACTGGA
ATCGCGTGATTCTGGAAGGGCGTAAGCCGGGCCTGACTTTGGGTATTGGCTGCGAAACGGCACAGTTCCC
GCTGACCAAAGTCGGCAAAGATCTGTTCCGCGATCTGAAGCGTGTTGCCCGCACAATGGACAGCATTGAC
GGCGGCGACGCGTATCAACAGATCTGCGACCAGCTGGTGGAATGTTTTGATAATCCGGAACTGACGTTCT
CAGCGCGTATTCTGCGTTCTATGATCGATCAGGGCATTGGCGGCACGGGGCGTTCACTCTCAGCGGAGTA
TCGTGAGATGCTGATGCAGGAGCCGTTAGAAGTGCTGAGCGAAGCAGATTTTGCCGCGGAACGCGATGCG
TCCGTGGTGCGTCAGAAAGAGGTTGAAGCGGCGGATACCGAGTCGTTTGAGGCGTTTCTGGCGAAGCAGT
CTTAA

>czcD
ATGAGCACCAATCACTCCCATGCCCTACCCAGCGAGCGCAATGAAAAATCGCTCTGGATCGCCCTGGCGC
TGACCGGAAGCTTCCTGCTGGCCGAAGTCGCCGCAGGCATTGTTTTCAACAGCCTGGCCCTGCTCTCAGA
TGCGGCGCACATGTTCACGGATGCAGCGGCGCTGGCCATCGCCTTGGCGGCTGTACGGGTGGCGAGACGG
CCAGCCGATGCCAAGCGCACCTATGGCTACTACCGCTTCGAAATCCTTGCGGCCGCATTCAACGCCCTGT
TGCTCTTCGGCGTCGCTGGCTACATCCTGTATGAGGGCTGGCAGCGTTTGCGCGAGCCAGCGCAGGTTCA
GTCCAGCGGGATGCTGGTGGTAGCAGTGCTGGGTCTGATCGTCAACCTCATCAGCATGCGTATGCTTCAG
GGCGGCAAGGACAAGAGCCTCAACGTCAAAGGCGCTTATCTGGAGGTCTGGAGCGACATGCTTGGCTCCA
TCGGCGTGATCGGAGCTGCGCTGGTCATCCGATTCACCGGCTGGGTGTGGGTAGATGCGGTCGTCGCCGT
GGCAATTGGGCTTTGGGTGCTGCCACGAACTTGGCTGCTACTCAAGGAAAGCCTGAATGTCTTGCTGGAG
GGAGTCCCAGACGGCATTGACATGGGAGCCTTGATGGCATCTGTTCAGGCCGTGCCAGGCGTGACAGGGC
TGCATGACTTGCACGTATGGGCGCTCACGAGCGGCAAGGTCAGCATGACCGCTCATATCGTGTATGCGCC
GGAAATAGGGCCCGACGAGTTGCTCAAAACCCTCAAGTTGTTGCTGGCTGACCGATTCAGCATCTTCCAT
ACGACGCTGCAACTGGAGATTGTTGCCTGTGAGCACAGCGTCGATGGGTGCAACTTCGTGGATCATCGCC
ACAAAGAACCGAGTTCTCAACCAGACCAAGCCGCAGCCCGGCAGGCGTGA

>cadR
ATGCGCATTGGTCAGTTGGCGCAGTTGGTAGGGGTCGAAACACAGACGATCCGCTTCTATGAACAGCAGG
GCTTGTTGCCGCCGCCTGATCGGCAGGACAACGGTTACCGTGTCTATACCGAGAAGCATGGTGAGGGGCT
GGCCTTCATCCGTCGCTGCAGAATCCTGGGCCTGTCACTGGCTGAGATTCACGAACTACAGAGCTATCAG
GACGACCCTCATCAGCCTTGTACCGCCGTCAACGCCTTGCTCGATGATCACATCTCTCATGTGCGGTCGC
AGATAACCGCTCTGCAAGCGCTTGAGAAACAACTCGTTTCACTGAGAGCGAGTTGCAACGATGACCGGGA
AGTTGAGGCGTGTGGGGTTCTTGCTGGAATTAGCGAAGGAAACATGCACCAGCAGTAG

>phoA
GTGAAAAAGTTTGTGAAAAAAGCATGGCCATTTGCAGTTGTGGCATCATTAGCAGTTACTTCGGTAGCAA
CATGGAATTTTACTCGTTCTAGTGAGGTTAATGCAGATGAGAACGGAAGCAATGCAAAGATTAAAAATGT
TATCGTATTAATTGGGGATGGTATGGGTCCTTCATATATGACGGCTCACCGTTATATGAAAGATAATCCA
AAAACCTTTGAAATGGAATCTACAGAATTTGATAAGCATCTTGTAGGAACACAAAAAACATACCCAGAAG
ATGAACATGAAAATATTACAGACTCTGCATCGGCAGCAACAGCTATGTCAGCAGGGATTAAAACATATAA
TGCTGCAATTGCAGTTGATAATGATAAAGCGGAAGTGAAAACAGTTCTTGAGCAAGCGAAAGAAAAAGGG
AAATCAACGGGACTCGTTGCAACTTCTGAAATTACACATGCAACACCAGCTGCTTTCGGTGCGCATGACA
TTAGCCGTAAAAATATGGACGCAATTGCAAATGATTATTTTGATGAGAAAATTAAAGGGAAGCATAAAGT
AGATGTTATGCTTGGCGGCGGCGTAAAGAACTTTGTTAGAAAAGATCGTAATCTTACAGAAGAGTTTAAG
AAATCTGGTTATAGCTATGTAACGGATCGAGATCAATTGTTAAATGATAAAAATGACCAAATTCTTGGTT
TATTTGCACCAGGTGGATTAGATAAAATGATTGACCGTAATGAGAAAACACCTTCATTAGAAGAAATGAC
TAATGCAGCAATTGAGCGTTTGAACAAAAATAAAAATGGTTTCTTTTTAATGGTGGAAGGAAGTCAAATC
GACTGGGCTGGACATGATAATGATATTGTTGGTGCAATGAGTGAAATGGAAGACTTCGAAAAAGCTTTCA
AAGCAGCGATTGAGTTTGCGAAAAAAGATAAAAATACGTTAGTTGTTGCAACGGCAGATCACGCTACTGG
TGGCTTATCTTTAGGTGCCAATGGCGAGTATAACTTTAAAGTAGAACCAATTAAAGCTGCAAAGCGCACA
CCAGACTTTATGGCAAATGAGATTGCAAAAGGTGCAAATGTAGAAGAAACATTGAAAAAGTATATTGATT
TACAATTAACTCCAGAAGAAATTCAAGCTGTAAATGATATAGCTCCGTCAAAAGATGTAACAAAGATTGA
TAATGCAATTGAAGATATTTTCAATAAGCGCTCGGTTACAGGCTGGACAACGGGTGGACATACAGGGGAA
GATGTGAACGTATATGCATTTGGTCCAGGTAAATACTTGTTCTCTGGCGTTCAAGAGAATACAAATATAG
CGAAACGTGTCTTTGATATTGTTGGCGGTGGCGATCCGAATAAAGGAAGACGCTAA
"""
# Esto crea el archivo automáticamente en Colab
with open("Genes_resistencia_Suelo.fasta", "w") as f:
    f.write(contenido_genes.strip())

print("Archivo 'Genes_resistencia_Suelo.fasta' creado correctamente.")

In [ ]:
def generar_reporte_exhaustivo():
    """
    Ejecuta BLASTn contra todos los genomas de la carpeta './genomas',
    calcula la identidad y cobertura respecto a los genes objetivo,
    y guarda un reporte detallado sin descartar ningún genoma.
    """
    archivo_genes = "Genes_resistencia_Suelo.fasta"
    directorio_genomas = "./genomas"
    archivo_salida_completo = "reporte_completo_13_genomas.csv"

    UMBRAL_IDENTIDAD = 70.0  # Identidad nucleotídica mínima (%)
    UMBRAL_COBERTURA = 80.0  # Cobertura de longitud del gen mínima (%)

    mapa_especies = {
        "NZ_CP045110.1": "Acinetobacter baumannii",
        "NZ_CP196955.1": "Acinetobacter calcoaceticus",
        "NZ_CP020102.1": "Bacillus subtilis",
        "NZ_OW968328.1": "Enterobacter cloacae",
        "NC_000913.3": "Escherichia coli",
        "NC_002516.2": "Pseudomonas aeruginosa",
        "NZ_LT907842.1": "Pseudomonas fluorescens",
        "NC_021505.1": "Pseudomonas putida",
        "NZ_AP024722.1": "Pseudomonas stutzeri",
        "NZ_CP104290.1": "Stenotrophomonas maltophilia",
        "NC_003197.2": "Salmonella enterica",
        "NC_007795.1": "Staphylococcus aureus",
        "NZ_CP034551.1": "Bacillus cereus"
    }

    if not os.path.exists(archivo_genes):
        print(f"Error: No se encontró el archivo '{archivo_genes}'. Crea el archivo FASTA antes de continuar.")
        return

    dict_longitudes = {}
    for record in SeqIO.parse(archivo_genes, "fasta"):
        dict_longitudes[record.id] = len(record.seq)

    archivos_genomas = glob.glob(os.path.join(directorio_genomas, "*"))
    archivos_genomas = [f for f in archivos_genomas if f.endswith(('.fasta', '.fa', '.fna'))]

    if not archivos_genomas:
        print(f"Error: No se encontraron archivos FASTA en '{directorio_genomas}'. Verifica los archivos subidos.")
        return

    columnas_outfmt = "qseqid sseqid pident length evalue bitscore"
    nombres_cols = ["Gen", "Subject", "Identidad", "Longitud_Alineamiento", "E_value", "Bit_Score"]

    todos_los_resultados = []

    print(f"Iniciando análisis detallado en {len(archivos_genomas)} genomas...\n")

    for idx, ruta_genoma in enumerate(archivos_genomas, 1):
        nombre_archivo = os.path.basename(ruta_genoma)
        especie = os.path.splitext(nombre_archivo)[0]

        for clave, nombre_especie in mapa_especies.items():
            if clave in nombre_archivo:
                especie = nombre_especie
                break

        db_temp = f"temp_db_{idx}"
        out_temp = f"temp_out_{idx}.tsv"

        !makeblastdb -in "{ruta_genoma}" -dbtype nucl -out "{db_temp}" > /dev/null
        !blastn -query "{archivo_genes}" -db "{db_temp}" -out "{out_temp}" -outfmt "6 {columnas_outfmt}"

        if os.path.exists(out_temp) and os.path.getsize(out_temp) > 0:
            df_temp = pd.read_csv(out_temp, sep="\t", names=nombres_cols)
            df_temp["Longitud_Gen_Total"] = df_temp["Gen"].map(dict_longitudes)
            df_temp["Cobertura"] = (df_temp["Longitud_Alineamiento"] / df_temp["Longitud_Gen_Total"]) * 100

            df_temp["Cumple_Criterio"] = df_temp.apply(
                lambda r: "Sí" if (r["Identidad"] >= UMBRAL_IDENTIDAD and r["Cobertura"] >= UMBRAL_COBERTURA) else "No (fuera de umbral)",
                axis=1
            )

            df_temp["Especie (origen del genoma)"] = especie
            todos_los_resultados.append(df_temp)
        else:
            df_vacio = pd.DataFrame([{
                "Especie (origen del genoma)": especie,
                "Gen": "Sin coincidencias",
                "Identidad": 0.0,
                "Cobertura": 0.0,
                "E_value": "N/A",
                "Cumple_Criterio": "Sin coincidencia detectada"
            }])
            todos_los_resultados.append(df_vacio)

        for ext in [".nhr", ".nin", ".nsq"]:
            if os.path.exists(db_temp + ext):
                os.remove(db_temp + ext)
        if os.path.exists(out_temp):
            os.remove(out_temp)

    if todos_los_resultados:
        df_final = pd.concat(todos_los_resultados, ignore_index=True)
        df_final["% Identidad"] = df_final["Identidad"].apply(lambda x: round(x, 1) if isinstance(x, (int, float)) else x)
        df_final["% Cobertura del gen"] = df_final["Cobertura"].apply(lambda x: round(x, 1) if isinstance(x, (int, float)) else x)

        cols_mostrar = ["Especie (origen del genoma)", "Gen", "% Identidad", "% Cobertura del gen", "E_value", "Cumple_Criterio"]
        df_reporte = df_final[cols_mostrar].rename(columns={"E_value": "E-value", "Cumple_Criterio": "Criterio (≥70% Id, ≥80% Cov)"})

        df_reporte = df_reporte.sort_values(by=["Especie (origen del genoma)"])
        df_reporte.to_csv(archivo_salida_completo, index=False)
        print(f"¡Análisis completo finalizado! Se guardó exitosamente como '{archivo_salida_completo}'.\n")

        pd.set_option('display.max_rows', None)
        display(df_reporte)

print("Función 'generar_reporte_exhaustivo' cargada exitosamente en la memoria.")

In [ ]:
# Llamar a la función principal definida en la Sección 3
generar_reporte_exhaustivo()

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def generar_mapa_calor_perfecto():
    """Lee el reporte CSV de BLASTn y genera el mapa de calor completo,

    corrigiendo los márgenes del eje X y la proporción visual.
    """
    archivo_csv = "reporte_completo_13_genomas.csv"

    if not os.path.exists(archivo_csv):
        print(f"Error: No se encontró el archivo '{archivo_csv}'.")
        return

    # 1. Cargar el archivo CSV
    df = pd.read_csv(archivo_csv)

    # Listas explícitas para ambos ejes
    lista_especies_referencia = [
        "Acinetobacter baumannii",
        "Acinetobacter calcoaceticus",
        "Bacillus cereus",
        "Bacillus subtilis",
        "Enterobacter cloacae",
        "Escherichia coli",
        "Pseudomonas aeruginosa",
        "Pseudomonas fluorescens",
        "Pseudomonas putida",
        "Pseudomonas stutzeri",
        "Salmonella enterica",
        "Staphylococcus aureus",
        "Stenotrophomonas maltophilia",
    ]

    lista_genes_estudio = [
        "Sul2",
        "Sul1",
        "blaTEM-1",
        "tetC",
        "floR",
        "cmlA",
        "blaOXA-1",
        "gshA",
        "czcD",
        "cadR",
        "phoA",
    ]

    # 2. Filtrar hits válidos
    col_criterio = "Criterio (≥70% Id, ≥80% Cov)"
    df_filtrado = df[df[col_criterio] == "Sí"].copy()

    # 3. Limpieza de texto y conversión numérica
    df_filtrado["Especie (origen del genoma)"] = (
        df_filtrado["Especie (origen del genoma)"]
        .astype(str)
        .str.replace("_", " ")
        .str.strip()
    )
    df_filtrado["Gen"] = df_filtrado["Gen"].astype(str).str.strip()
    df_filtrado["% Identidad"] = pd.to_numeric(
        df_filtrado["% Identidad"], errors="coerce"
    )

    # 4. Crear la matriz 2D
    df_pivot = df_filtrado.pivot_table(
        index="Especie (origen del genoma)",
        columns="Gen",
        values="% Identidad",
        aggfunc="max",
    )

    # Reindexar ambos ejes para asegurar la presencia de todos los elementos
    df_pivot = df_pivot.reindex(
        index=lista_especies_referencia, columns=lista_genes_estudio
    )

    # 5. Configurar la figura con espacio amplio para el eje X
    plt.figure(figsize=(12, 8))
    sns.set_theme(style="whitegrid")

    # Dibujar el Mapa de Calor
    ax = sns.heatmap(
        df_pivot,
        annot=True,  # Mostrar los valores numéricos
        fmt=".1f",  # Un decimal
        cmap="YlOrRd",  # Gradiente de amarillo a rojo
        vmin=70,  # Límite inferior
        vmax=100,  # Límite superior
        linewidths=1,
        linecolor="white",
        cbar_kws={"label": "Porcentaje (%) de Identidad Nucleotídica"},
    )

    # Color gris claro para casillas sin hits
    ax.set_facecolor("#efefef")

    # 6. Personalizar Títulos y Ejes
    plt.title(
        "Mapa de Calor: %Identidad de genes de resistencia\n(solo hits con cobertura del gen >= 80%)\n",
        fontsize=13,
        fontweight="bold",
    )
    plt.xlabel("Genes de Resistencia Evaluados", fontsize=11, labelpad=12)
    plt.ylabel("Genomas de referencia (NCBI RefSeq)", fontsize=11, labelpad=12)

    # AJUSTE CLAVE: Asegurar la visibilidad de los nombres en el eje X
    plt.xticks(
        rotation=45,
        ha="right",
        rotation_mode="anchor",
        fontsize=10,
        fontweight="normal",
    )
    plt.yticks(rotation=0, fontsize=10, fontstyle="italic")

    # Ajustar márgenes dinámicamente para que no se corte nada
    plt.tight_layout()

    # Guardar la imagen
    plt.savefig("mapa_calor_estético.png", dpi=300, bbox_inches="tight")
    print(
        "Gráfica ajustada con éxito y guardada como 'mapa_calor_estético.png'"
    )

    plt.show()


# Ejecutar la función
generar_mapa_calor_perfecto()